In [1]:
!rm -rf cache/validation/

In [2]:
import os
from multiprocess import set_start_method
set_start_method("spawn")
import logging
logger = logging.getLogger("ignite.handlers.early_stopping.EarlyStopping")
logger.setLevel(logging.WARNING)

In [3]:
from transformers import Trainer, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from coreset_trainer.trainer import CoresetTrainer
import os
import torch

import argparse



In [4]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types
)
train_dataset, test_dataset, estimators, test_indices = load_data_and_estimators()


Repo card metadata block was not found. Setting CardData to empty.
[WARNING] Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.
[WARNING] Repo card metadata block was not found. Setting CardData to empty.


influence_estimate_path: ./results/influence/LESSEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test/estimate.parquet
dirname: ./results/influence/LESSEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test
exists: True
influence_estimate_path: ./results/influence/DataInfEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test/estimate.parquet
dirname: ./results/influence/DataInfEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test
exists: True


In [5]:
from functools import partial
from explanations import KRandom
k_random_types = [partial(KRandom, seed=s) for s in range(9)]
explanation_types = explanation_types + k_random_types

In [6]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from tqdm import tqdm
import itertools
import pandas as pd
import traceback

logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')
multiprocessing.set_start_method('spawn', force=True)
torch.manual_seed(42)




num_devices = torch.cuda.device_count()







In [7]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from validation_worker_validation import process

logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')

multiprocessing.set_start_method('spawn', force=True)

num_gpus = torch.cuda.device_count()
logging.info(f"Detected {num_gpus} GPUs")


import warnings
warnings.filterwarnings("ignore")

[INFO] Detected 2 GPUs


In [ ]:

device_ids = itertools.cycle(range(num_devices))
results = []

from peft import LoraConfig, get_peft_model, PeftModel


for estimator in estimators:
    

    explanations = [
        explanation_type(row.name, estimator)
        for explanation_type in explanation_types
        for _, row in estimator.influence_estimate.iloc[test_indices].iterrows()
    ]
            
        
    partial_results_dir =  os.path.join("./cache/validation/partial/",
        estimator.get_config_string(),
        os.path.basename(estimator.model_path),
        train_dataset_name,
        train_dataset_split,
        test_dataset_name,
        test_dataset_split,
        "partial")
    with ProcessPoolExecutor(max_workers=2*num_devices) as executor:
        futures = {}
        for seed in range(1):
            for ii, explanation in enumerate(explanations):
                
                # # random documents to test log_p for in addition to the document beeing explained
                # random_test_indices = test_dataset.shuffle(seed=ii).shuffle(seed=seed).select(range(explanation.k))["indices"]
            
                text_indices = [explanation.document_idx] #+ random_test_indices
                futures[explanation] = executor.submit(
                        process,
                        partial_results_dir,
                        estimator, explanation,
                        train_dataset.select(explanation.documents), explanation.documents, 
                        test_dataset.select(text_indices), text_indices,                 
                        train_dataset, train_dataset_name, train_dataset_split, test_dataset, test_dataset_name, test_dataset_split, 
                        next(device_ids),
                        seed,
                        ii
                )

        with tqdm(total=len(futures), desc="Explanations", position=0) as pbar:
            for future in as_completed(futures.values()):
                try:
                    future.result()  
                except Exception as e:
                    logging.error(f"A future failed: {e}\n{traceback.format_exc()}")
                    raise
                finally:
                    pbar.update(1)

Explanations:   0%|          | 0/1400 [00:00<?, ?it/s]No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set a

{'train_runtime': 13.5474, 'train_samples_per_second': 18.454, 'train_steps_per_second': 18.454, 'train_loss': 3.02781494140625, 'epoch': 25.0}
{'train_runtime': 12.9965, 'train_samples_per_second': 19.236, 'train_steps_per_second': 19.236, 'train_loss': 3.02715966796875, 'epoch': 25.0}
{'train_runtime': 13.04, 'train_samples_per_second': 19.172, 'train_steps_per_second': 19.172, 'train_loss': 3.00219873046875, 'epoch': 25.0}
{'train_runtime': 13.0598, 'train_samples_per_second': 19.143, 'train_steps_per_second': 19.143, 'train_loss': 3.00130859375, 'epoch': 25.0}
{'train_runtime': 12.9841, 'train_samples_per_second': 19.254, 'train_steps_per_second': 19.254, 'train_loss': 3.054629150390625, 'epoch': 25.0}
{'train_runtime': 12.7992, 'train_samples_per_second': 19.532, 'train_steps_per_second': 19.532, 'train_loss': 3.04400830078125, 'epoch': 25.0}
{'train_runtime': 12.7774, 'train_samples_per_second': 19.566, 'train_steps_per_second': 19.566, 'train_loss': 3.024914794921875, 'epoch': 2

Explanations:  16%|█▌        | 225/1400 [13:00<50:16,  2.57s/it]  No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Explanations:  16%|█▌        | 226/1400 [13:11<1:42:39,  5.25s/it]

{'train_runtime': 13.5407, 'train_samples_per_second': 18.463, 'train_steps_per_second': 18.463, 'train_loss': 3.034010498046875, 'epoch': 25.0}
{'train_runtime': 12.9388, 'train_samples_per_second': 19.322, 'train_steps_per_second': 19.322, 'train_loss': 3.032939208984375, 'epoch': 25.0}
{'train_runtime': 12.957, 'train_samples_per_second': 19.295, 'train_steps_per_second': 19.295, 'train_loss': 3.035398681640625, 'epoch': 25.0}
{'train_runtime': 12.9882, 'train_samples_per_second': 19.248, 'train_steps_per_second': 19.248, 'train_loss': 3.035988525390625, 'epoch': 25.0}
{'train_runtime': 12.9336, 'train_samples_per_second': 19.329, 'train_steps_per_second': 19.329, 'train_loss': 3.014712158203125, 'epoch': 25.0}
{'train_runtime': 12.9573, 'train_samples_per_second': 19.294, 'train_steps_per_second': 19.294, 'train_loss': 3.036077392578125, 'epoch': 25.0}
{'train_runtime': 12.8537, 'train_samples_per_second': 19.45, 'train_steps_per_second': 19.45, 'train_loss': 3.01635791015625, 'epo

Explanations:  16%|█▌        | 227/1400 [13:12<1:15:14,  3.85s/it]No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Explanations:  16%|█▋        | 228/1400 [13:13<59:51,  3.06s/it]  

...

Explanations:  49%|████▉     | 686/1400 [38:33<40:03,  3.37s/it]No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


{'train_runtime': 11.256, 'train_samples_per_second': 22.21, 'train_steps_per_second': 22.21, 'train_loss': 2.83835888671875, 'epoch': 25.0}
{'train_runtime': 11.3482, 'train_samples_per_second': 22.03, 'train_steps_per_second': 22.03, 'train_loss': 2.86449462890625, 'epoch': 25.0}
{'train_runtime': 11.4115, 'train_samples_per_second': 21.908, 'train_steps_per_second': 21.908, 'train_loss': 2.872567138671875, 'epoch': 25.0}
{'train_runtime': 11.2597, 'train_samples_per_second': 22.203, 'train_steps_per_second': 22.203, 'train_loss': 2.861055908203125, 'epoch': 25.0}
{'train_runtime': 11.3216, 'train_samples_per_second': 22.082, 'train_steps_per_second': 22.082, 'train_loss': 2.838092041015625, 'epoch': 25.0}
{'train_runtime': 11.4273, 'train_samples_per_second': 21.877, 'train_steps_per_second': 21.877, 'train_loss': 2.85809765625, 'epoch': 25.0}
{'train_runtime': 11.1644, 'train_samples_per_second': 22.393, 'train_steps_per_second': 22.393, 'train_loss': 2.851984375, 'epoch': 25.0}
{'

Explanations:  49%|████▉     | 687/1400 [38:38<45:35,  3.84s/it]

{'train_runtime': 11.5258, 'train_samples_per_second': 21.69, 'train_steps_per_second': 21.69, 'train_loss': 2.848899169921875, 'epoch': 25.0}
{'train_runtime': 11.6918, 'train_samples_per_second': 21.382, 'train_steps_per_second': 21.382, 'train_loss': 2.849257568359375, 'epoch': 25.0}
{'train_runtime': 11.5796, 'train_samples_per_second': 21.59, 'train_steps_per_second': 21.59, 'train_loss': 2.870759521484375, 'epoch': 25.0}
{'train_runtime': 11.5103, 'train_samples_per_second': 21.72, 'train_steps_per_second': 21.72, 'train_loss': 2.865728515625, 'epoch': 25.0}
{'train_runtime': 11.4073, 'train_samples_per_second': 21.916, 'train_steps_per_second': 21.916, 'train_loss': 2.8734697265625, 'epoch': 25.0}
{'train_runtime': 11.1908, 'train_samples_per_second': 22.34, 'train_steps_per_second': 22.34, 'train_loss': 2.8558583984375, 'epoch': 25.0}
{'train_runtime': 11.4108, 'train_samples_per_second': 21.909, 'train_steps_per_second': 21.909, 'train_loss': 2.85898388671875, 'epoch': 25.0}
{

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Explanations:  50%|████▉     | 696/1400 [39:06<36:48,  3.14s/it]No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Explanations:  50%|█████     | 700/1400 [39:20<37:04,  3.18s/it]No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base model

In [ ]:
import pandas as pd

df_validation = pd.read_parquet("./cache/validation/partial/")

In [ ]:
[r for r in df_validation.explanation_type.unique() if "random examples with" in r]

['10 random examples with seed 42',
 '10 random examples with seed 0',
 '10 random examples with seed 1',
 '10 random examples with seed 2',
 '10 random examples with seed 3',
 '10 random examples with seed 4',
 '10 random examples with seed 5',
 '10 random examples with seed 6',
 '10 random examples with seed 7',
 '10 random examples with seed 8']

In [ ]:
import pandas as pd
all_rows = []
for rand_col in [r for r in df_validation.explanation_type.unique() if "random examples with" in r]:
    group_cols = ["document_idx", "train_dataset", "train_split", "test_dataset","test_split",
                "model", "estimator"]


    baseline_df = df_validation[df_validation["explanation_type"] == rand_col].copy()
    


    for expl_type in df_validation["explanation_type"].unique():
        expl_df = df_validation[df_validation["explanation_type"] == expl_type].copy()
        merged = expl_df.merge(
            baseline_df[group_cols + ["indices_target_document", "delta_target_document"]],
            on=group_cols + ["indices_target_document"],
            suffixes=("", "_random")
        )
        
        # 1(Δ_S > Δ_R)
        merged["xi"] = (merged["delta_target_document"] >= merged["delta_target_document_random"]).astype(int)
        merged["random_run"] = rand_col



        all_rows.append(merged[group_cols + ["explanation_type", "indices_target_document", 
                                            "delta_target_document", "delta_target_document_random", "xi", "random_run"]])


xi_df = pd.concat(all_rows, ignore_index=True)

In [ ]:
xi_df["explanation_type"] = xi_df.apply(
    lambda row: "rand" if "random " in row["explanation_type"] else row["explanation_type"], axis=1
)

In [ ]:
xi_df.groupby(["explanation_type","train_dataset", "train_split", "test_dataset","test_split",
              "model", "estimator"])["xi"].describe()

count  \
explanation_type                                   train_dataset              train_split test_dataset               test_split model                                estimator                                           
Top-10 least influential (scores closest to zero)  loris3/tulu-v2-sft-mixture test        loris3/tulu-v2-sft-mixture test       pythia-31m_tulu-v2-sft-mixture_train DataInfEstimator: fast_implementation=True   50.0   
                                                                                                                                                                     LESSEstimator: normalize=True                50.0   
Top-10 most harmful (most positive scores)         loris3/tulu-v2-sft-mixture test        loris3/tulu-v2-sft-mixture test       pythia-31m_tulu-v2-sft-mixture_train DataInfEstimator: fast_implementation=True   50.0   
                                                                                                                                                                     LESSEstimator: normalize=True                50.0   
Top-10 most helpful (most negative scores)         loris3/tulu-v2-sft-mixture test        loris3/tulu-v2-sft-mixture test       pythia-31m_tulu-v2-sft-mixture_train DataInfEstimator: fast_implementation=True   50.0   
                                                                                                                                                                     LESSEstimator: normalize=True                50.0   
Top-10 most influential (scores with largest ab... loris3/tulu-v2-sft-mixture test        loris3/tulu-v2-sft-mixture test       pythia-31m_tulu-v2-sft-mixture_train DataInfEstimator: fast_implementation=True   50.0   
                                                                                                                                                                     LESSEstimator: normalize=True                50.0   
rand                                               loris3/tulu-v2-sft-mixture test        loris3/tulu-v2-sft-mixture test       pythia-31m_tulu-v2-sft-mixture_train DataInfEstimator: fast_implementation=True  500.0   
                                                                                                                                                                     LESSEstimator: normalize=True               500.0   

                                                                                                                                                                                                                 mean  \
explanation_type                                   train_dataset              train_split test_dataset               test_split model                                estimator                                          
Top-10 least influential (scores closest to zero)  loris3/tulu-v2-sft-mixture test        loris3/tulu-v2-sft-mixture test       pythia-31m_tulu-v2-sft-mixture_train DataInfEstimator: fast_implementation=True  0.42   
                                                                                                                                                                     LESSEstimator: normalize=True               0.70   
Top-10 most harmful (most positive scores)         loris3/tulu-v2-sft-mixture test        loris3/tulu-v2-sft-mixture test       pythia-31m_tulu-v2-sft-mixture_train DataInfEstimator: fast_implementation=True  0.30   
                                                                                                                                                                     LESSEstimator: normalize=True               0.50   
Top-10 most helpful (most negative scores)         loris3/tulu-v2-sft-mixture test        loris3/tulu-v2-sft-mixture test       pythia-31m_tulu-v2-sft-mixture_train DataInfEstimator: fast_implementation=True  0.40   
                                                                         

In [ ]:
df_scoring = pd.read_parquet("./cache/scoring/partial/")

In [ ]:
df = df_scoring.merge(
    df_validation,
    on=["explanation_type", "model", "estimator", "document_idx", "train_dataset", "train_split", "test_dataset", "test_split"],
    how="left"
)[["explanation_type", "model", "linear_coder", "estimator", "document_idx", "pred_gain","mse", "delta_target_document"]]

In [ ]:
import statsmodels.api as sm
import pandas as pd

results = []

for keys, group in df.groupby(["explanation_type", "model", "estimator", "linear_coder"]):
    x = -pd.to_numeric(group["mse"], errors="coerce")  # negate MSE so lower is better
    y = pd.to_numeric(group["delta_target_document"], errors="coerce")
    
    mask = ~(x.isna() | y.isna())
    x_clean, y_clean = x[mask], y[mask]
    

    X = sm.add_constant(pd.DataFrame({"neg_mse": x_clean}))
    model = sm.OLS(y_clean, X).fit()
    
    results.append({
        "explanation_type": keys[0],
        "model": keys[1],
        "estimator": keys[2],
        "linear_coder": keys[3],
        "coef": model.params.get("neg_mse", None),       
        "p_value": model.pvalues.get("neg_mse", None),  
        "r_squared": model.rsquared
    })

results_df = pd.DataFrame(results)
results_df


,explanation_type,model,estimator,linear_coder,coef,p_value,r_squared
0,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,KLTCoder,-2.570608,0.510432,0.156053
1,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,MSECoder,-2.573869,0.509932,0.156391
2,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,MSECoderNNLSL2,-2.560777,0.505799,0.159201
3,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,MSECoderProjUSimp,-2.657760,0.486983,0.172374
4,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,MSECoderProjUSimpSparse,-2.713055,0.487209,0.172212
...,...,...,...,...,...,...,...
61,Top-10 most influential (scores with largest a...,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,MSECoderLemon,-0.319639,0.905681,0.005498
62,Top-10 most influential (scores with largest a...,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,MSECoderNNLSL2,-0.231939,0.911148,0.004878
63,Top-10 most influential (scores with largest a...,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,MSECoderProjUSimp,-0.301747,0.907474,0.005290
64,Top-10 most influential (scores with largest a...,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,MSECoderProjUSimpSparse,-0.314037,0.907163,0.005326


In [ ]:
import statsmodels.api as sm

results = []


for keys, group in df.groupby(["explanation_type", "model", "estimator", "linear_coder"]):
    x = pd.to_numeric(group["pred_gain"], errors="coerce")
    y = pd.to_numeric(group["delta_target_document"], errors="coerce")
    
    mask = ~(x.isna() | y.isna())
    x_clean, y_clean = x[mask], y[mask]
    

    X = sm.add_constant(x_clean)
    model = sm.OLS(y_clean, X).fit()
    
    results.append({
        "explanation_type": keys[0],
        "model": keys[1],
        "estimator": keys[2],
        "linear_coder": keys[3],
        "coef": model.params.get("pred_gain", None),
        "p_value": model.pvalues.get("pred_gain", None),
        "r_squared": model.rsquared
    })


results_df = pd.DataFrame(results)
results_df


,explanation_type,model,estimator,linear_coder,coef,p_value,r_squared
0,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,KLTCoder,-2.830133e+03,0.296353,0.346538
1,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,MSECoder,-3.191615e+02,0.232349,0.426176
2,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,MSECoderNNLSL2,-2.607489e+02,0.237516,0.419232
3,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,MSECoderProjUSimp,-2.715393e+03,0.007453,0.933358
4,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,MSECoderProjUSimpSparse,-1.393003e+03,0.212512,0.453792
...,...,...,...,...,...,...,...
61,Top-10 most influential (scores with largest a...,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,MSECoderLemon,3.072965e-02,0.053357,0.761921
62,Top-10 most influential (scores with largest a...,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,MSECoderNNLSL2,3.705363e-08,0.055823,0.755056
63,Top-10 most influential (scores with largest a...,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,MSECoderProjUSimp,3.705659e-08,0.055752,0.755253
64,Top-10 most influential (scores with largest a...,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,MSECoderProjUSimpSparse,3.706277e-08,0.055752,0.755252
